In [1]:
%pip install -r ../requirements.txt

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached notebook-7.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached jupyter_console-6.6.3-py3-none-any.whl.metadata (5.8 kB)
  Using cached nbconvert-7.17.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached jupyterlab-4.6.0-py3-none-any.whl.metadata (16 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
  Using cached async_lru-2.3.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jupyter_builder-1.0.2-py3-none-any.whl.metadata (7.7 kB)
  Using cached jupyter_lsp-2.3.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached jupyter_server-2.20.0-py3

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [2]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import folium
from folium.plugins import MarkerCluster

In [3]:
data = pd.read_excel('../data/raw/active_points_Other.xlsx', engine="openpyxl", skiprows=9, header=None,
    names=["Тип точки", "Регион", "Город", "address_raw", "Техническое название", "working_hours"])

In [4]:
data = data[data["Город"] == "Красноярск"].reset_index(drop=True)
data['id'] = range(len(data))

data = data[["address_raw", "working_hours"]]

data['address_clean'] = data['address_raw'].copy()

In [5]:
expressions = {"  ": " ", "Россия": "", "Край": "", "Красноярский": "", "Красноярск": "",  "д.": "", "ул.": "улица", "пер.": "",
                "Дом ": " ", "Строение": "", "2-Я": "2-я", "Им.": "Имени ", "Корпус": "", " Г. ": " "}

for old, new in expressions.items():
    data["address_clean"] = data["address_clean"].str.replace(old, new, regex=False)

In [6]:
data["address_clean"] = "Красноярск, " + data["address_clean"]

In [7]:
exceptional_names = {" няя ": " Крайняя ", "Газеты  Рабочий": "Газеты Красноярский Рабочий","Проспект Мира 19  1": "Проспект Мира 19",
                     "улица Елены Стасовой 80 С 1": "улица Елены Стасовой 80"}

for old, new in exceptional_names.items():
    data["address_clean"] = data["address_clean"].str.replace(old, new, regex=False)

In [8]:
data["address_clean"] = data["address_clean"].str.rsplit(n=1).str.join(", ")

In [9]:
geolocator = Nominatim(user_agent="task_1_(xlsx->locations_dataset)", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2, error_wait_seconds=10, swallow_exceptions=True)
tqdm.pandas(desc="Геокодирование адресов")
data["geocoding_quality"] = data["address_clean"].progress_apply(geocode)

Геокодирование адресов: 100%|██████████| 420/420 [07:01<00:00,  1.00s/it]


In [10]:
data["latitude"] = data["geocoding_quality"].apply(lambda x: x.latitude if x else None)
data["longitude"] = data["geocoding_quality"].apply(lambda x: x.longitude if x else None)

Почему-то улицы Академгородок нет в Nominatim, поэтому руками добавляем все ПВЗ на ней

In [11]:
def add_akademgorodok_to_data(index: int, latitude: float, longitude: float, house_number: int) -> None:
    data.loc[index, ["latitude", "longitude", "geocoding_quality"]] = [latitude, longitude,
                    ", ".join((str(house_number), "улица Академгородок, район Академгородок, Красноярск",
                     "городской округ Красноярск, Красноярский край, Сибирский федеральный округ, 660000, Россия"))]

add_akademgorodok_to_data(index=4, latitude=55.987347, longitude=92.781677, house_number=74)
add_akademgorodok_to_data(index=12, latitude=55.985685, longitude=92.752050, house_number=12)
add_akademgorodok_to_data(index=321, latitude=55.987382, longitude=92.774715, house_number=66)
add_akademgorodok_to_data(index=324, latitude=55.992835, longitude=92.757799, house_number=17)

In [14]:
from pathlib import Path
BASE_DIR = Path.cwd().parent

data = data.reindex(columns=["address_raw", "address_clean", "latitude", "longitude", "working_hours", "geocoding_quality"])
csv_path = BASE_DIR / 'data' / 'processed' / 'pvz_krasnoyarsk.csv'
data.to_csv(csv_path)

In [15]:
map = folium.Map(location=[data['latitude'].mean(), data['longitude'].mean()], zoom_start=11)
marker_cluster = MarkerCluster().add_to(map)
for _, row in data.iterrows():
    popup_text = f"<b>Адрес:</b> {row['address_clean']}<br> <b>Часы работы:</b> {row['working_hours']}<br> <b>Координаты:</b> {row['latitude']:.6f}, {row['longitude']:.6f}"
    folium.Marker(location=[row['latitude'], row['longitude']], popup=folium.Popup(popup_text, max_width=300), tooltip=row['address_clean']).add_to(marker_cluster)

map_path = BASE_DIR / 'reports' / 'pvz_krasnoyarsk_map.html'
map.save(map_path)